# Auto-Labeling Ground Truth & Instance Segmentation dengan `facebook/mask2former-swin-large-ade-panoptic`

Notebook ini menggunakan model pretrained [`facebook/mask2former-swin-large-ade-panoptic`](https://huggingface.co/facebook/mask2former-swin-large-ade-panoptic) untuk melakukan **Instance Segmentation** pada folder gambar mentah di Kaggle dan **menghasilkan dataset Ground Truth lengkap beserta `data.yaml`** secara otomatis (tanpa memerlukan `data.yaml` di folder input).

### Output yang Dihasilkan di `/kaggle/working/`:
1. `data.yaml` (di `/kaggle/working/data.yaml`, `/kaggle/working/outputs/data.yaml`, dan `/kaggle/working/outputs/dataset/data.yaml`) yang diturunkan langsung dari `id2label` model **Mask2Former Swin-Large**.
2. Dataset terstruktur di `/kaggle/working/outputs/dataset/{train,valid,test}/{images,labels,masks}`:
   - `images/`: salinan gambar per split (`train` / `valid` / `test`).
   - `labels/*.txt`: anotasi poligon instance segmentation (`class_id x1 y1 x2 y2 ...`).
   - `masks/*_mask.png`: instance mask PNG hasil Swin-Large.
3. Visualisasi prediksi di `/kaggle/working/outputs/vis/` dan ringkasan CSV di `/kaggle/working/outputs/instance_segmentation_results.csv`.
4. Arsip siap unduh `/kaggle/working/kaggle_segmentation_outputs.zip`.


## 1. Instalasi & Import Library


In [ ]:
import importlib.metadata
import importlib.util
import os
import sys
import subprocess
import numpy as np

constraint_lines = [f"numpy=={np.__version__}"]
for dist_name in ["scipy", "torch", "torchvision"]:
    try:
        constraint_lines.append(f"{dist_name}=={importlib.metadata.version(dist_name)}")
    except importlib.metadata.PackageNotFoundError:
        pass

os.makedirs("/kaggle/working", exist_ok=True) if os.path.exists("/kaggle") else None
constraint_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
constraint_path = os.path.join(constraint_dir, "pip_constraints.txt")
with open(constraint_path, "w", encoding="utf-8") as cf:
    cf.write("\n".join(constraint_lines) + "\n")

required_pkgs = [
    ("transformers", "transformers>=4.40.0"),
    ("accelerate", "accelerate>=0.28.0"),
    ("timm", "timm>=0.9.16"),
    ("yaml", "pyyaml>=6.0"),
]
missing_pkgs = [pkg for mod, pkg in required_pkgs if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", constraint_path, *missing_pkgs], check=False)


In [ ]:
import random
import shutil
import time
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device aktif : {device}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Konfigurasi Path Dataset Input (Tanpa Input `data.yaml`)
Notebook ini hanya membaca gambar dari `KAGGLE_DATASET_DIR`. Label `.txt`, `masks/`, dan file `data.yaml` akan dibuat otomatis dari output model Swin-Large.


In [ ]:
# ============================================================
# KONFIGURASI PATH & PARAMETER
# ============================================================
MODEL_NAME = "facebook/mask2former-swin-large-ade-panoptic"

KAGGLE_DATASET_DIR = "/kaggle/input/datasets/melekbengi/data-viskom/data viskom"
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") or os.access("/", os.W_OK) else os.path.abspath("./working")

OUTPUT_DIR = os.path.join(WORKING_DIR, "outputs")
DATASET_OUT_DIR = os.path.join(OUTPUT_DIR, "dataset")
VIS_DIR = os.path.join(OUTPUT_DIR, "vis")

CSV_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "instance_segmentation_results.csv")
DATA_YAML_WORKING = os.path.join(WORKING_DIR, "data.yaml")
DATA_YAML_OUTPUT = os.path.join(OUTPUT_DIR, "data.yaml")
DATA_YAML_DATASET = os.path.join(DATASET_OUT_DIR, "data.yaml")
ZIP_OUTPUT_BASE = os.path.join(WORKING_DIR, "kaggle_segmentation_outputs")

# Parameter inferensi & pembangkitan ground truth
SCORE_THRESHOLD = 0.50
MASK_THRESHOLD = 0.50
OVERLAP_THRESHOLD = 0.80
MIN_POLYGON_AREA_PX = 25.0
REMAP_TO_DETECTED_CLASSES = True
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # train, valid, test (dipakai jika input belum terpisah per split)
RANDOM_SEED = 42
MAX_VIS_IMAGES = 8
SAVE_ALL_VISUALIZATIONS = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASET_OUT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

print(f"Input Images Dir : {KAGGLE_DATASET_DIR}")
print(f"Working Dir      : {WORKING_DIR}")
print(f"Dataset Out Dir  : {DATASET_OUT_DIR}")
print(f"Zip Output       : {ZIP_OUTPUT_BASE}.zip")


In [ ]:
# ============================================================
# AUTO-DISCOVERY GAMBAR & PENENTUAN SPLIT (TRAIN / VALID / TEST)
# ============================================================
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def discover_images_with_splits(dataset_dir: str, ratios=(0.70, 0.15, 0.15), seed: int = 42):
    root = Path(dataset_dir)
    if not root.exists():
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            matches = [p for p in kaggle_root.rglob("*") if p.is_dir() and p.name == root.name]
            if matches:
                root = matches[0]
    if not root.exists():
        raise FileNotFoundError(
            f"Direktori dataset tidak ditemukan: {dataset_dir}. "
            "Pastikan dataset sudah ditambahkan ke Kaggle Notebook (+ Add Input)."
        )

    split_records = []
    split_aliases = [("train", ["train"]), ("valid", ["valid", "val"]), ("test", ["test"])]
    found_existing_split = False

    for split_name, aliases in split_aliases:
        for alias in aliases:
            candidate_dirs = []
            if (root / alias / "images").exists():
                candidate_dirs.append(root / alias / "images")
            elif (root / alias).exists():
                candidate_dirs.append(root / alias)
            else:
                candidate_dirs.extend(sorted(root.glob(f"**/{alias}/images")))
                if not candidate_dirs:
                    candidate_dirs.extend(sorted(root.glob(f"**/{alias}")))

            for cdir in candidate_dirs:
                imgs = sorted([p for p in cdir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
                if imgs:
                    found_existing_split = True
                    for p in imgs:
                        split_records.append((split_name, p))
                    break
            if found_existing_split and any(s == split_name for s, _ in split_records):
                break

    if not found_existing_split:
        all_imgs = sorted({p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS})
        rng = random.Random(seed)
        rng.shuffle(all_imgs)
        n_total = len(all_imgs)
        n_train = int(n_total * ratios[0])
        n_valid = int(n_total * ratios[1])

        for p in all_imgs[:n_train]:
            split_records.append(("train", p))
        for p in all_imgs[n_train:n_train + n_valid]:
            split_records.append(("valid", p))
        for p in all_imgs[n_train + n_valid:]:
            split_records.append(("test", p))

    return split_records

image_split_records = discover_images_with_splits(KAGGLE_DATASET_DIR, ratios=SPLIT_RATIOS, seed=RANDOM_SEED)
split_counts = pd.Series([s for s, _ in image_split_records]).value_counts().to_dict()
print(f"Total gambar ditemukan : {len(image_split_records)}")
print(f"Distribusi split       : {split_counts}")
for sp, sample in image_split_records[:5]:
    print(f" - [{sp}] {sample}")


## 3. Load Model `facebook/mask2former-swin-large-ade-panoptic`
Label kelas (`id2label`) akan diambil langsung dari konfigurasi pretrained model ini untuk menyusun `data.yaml`.


In [ ]:
print(f"Memuat processor dan model: {MODEL_NAME} ...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

ade_id2label = {int(k): str(v) for k, v in model.config.id2label.items()}
print(f"Model siap! Jumlah kelas ADE20K pada Swin-Large: {len(ade_id2label)}")


## 4. Fungsi Inferensi, Ekstraksi Poligon Ground Truth & Visualisasi


In [ ]:
def get_color_palette(seed: int = 42, num_colors: int = 256):
    rng = np.random.default_rng(seed)
    return rng.integers(50, 255, size=(num_colors, 3), dtype=np.uint8)

PALETTE = get_color_palette()

@torch.no_grad()
def predict_instances(
    image: Image.Image,
    score_threshold: float = SCORE_THRESHOLD,
    mask_threshold: float = MASK_THRESHOLD,
    overlap_threshold: float = OVERLAP_THRESHOLD,
):
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    use_amp = device.type == "cuda"
    with torch.cuda.amp.autocast(enabled=use_amp):
        outputs = model(**inputs)

    target_size = [(image.height, image.width)]
    result = processor.post_process_instance_segmentation(
        outputs,
        threshold=score_threshold,
        mask_threshold=mask_threshold,
        overlap_mask_area_threshold=overlap_threshold,
        target_sizes=target_size,
    )[0]

    seg_map = result["segmentation"].cpu().numpy()
    segments_info = result["segments_info"]
    return seg_map, segments_info


def mask_to_normalized_polygon(mask: np.ndarray, width: int, height: int, min_area: float = MIN_POLYGON_AREA_PX):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < min_area:
        return None
    peri = cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, 0.002 * peri, True)
    pts = approx.reshape(-1, 2).astype(np.float32)
    if len(pts) < 3:
        pts = largest.reshape(-1, 2).astype(np.float32)
    if len(pts) < 3:
        return None

    pts[:, 0] = np.clip(pts[:, 0] / float(width), 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / float(height), 0.0, 1.0)
    return pts.flatten().tolist()


def draw_instance_segmentation(
    image: Image.Image,
    seg_map: np.ndarray,
    segments_info: list,
    alpha: float = 0.45,
):
    img_np = np.array(image.convert("RGB"), dtype=np.uint8)
    overlay = img_np.copy()

    for idx, seg in enumerate(segments_info):
        inst_id = seg["id"]
        mask = seg_map == inst_id
        if not np.any(mask):
            continue

        color = PALETTE[idx % len(PALETTE)].tolist()
        overlay[mask] = (
            img_np[mask] * (1.0 - alpha) + np.array(color, dtype=np.float32) * alpha
        ).astype(np.uint8)

        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, color, 2)

        ys, xs = np.where(mask)
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        label_name = ade_id2label.get(int(seg["label_id"]), str(seg["label_id"]))
        text = f"{label_name} {seg['score']:.2f}"

        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        ty1 = max(0, y1 - th - 6)
        cv2.rectangle(overlay, (x1, ty1), (x1 + tw + 4, ty1 + th + 6), color, -1)
        cv2.putText(
            overlay, text, (x1 + 2, ty1 + th + 2),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA
        )

    return Image.fromarray(overlay)


## 5. Generate Ground Truth Dataset, `data.yaml` dari Swin-Large, dan File `.zip`


In [ ]:
for sp in ["train", "valid", "test"]:
    for sub in ["images", "labels", "masks"]:
        os.makedirs(os.path.join(DATASET_OUT_DIR, sp, sub), exist_ok=True)
    os.makedirs(os.path.join(VIS_DIR, sp), exist_ok=True)

raw_records = []
pending_label_files = []
detected_ade_ids = set()
visualized_samples = []

start_time = time.time()
for idx, (split_name, img_path) in enumerate(tqdm(image_split_records, desc="Auto-Labeling Swin-Large")):
    image = Image.open(img_path).convert("RGB")
    img_name = img_path.name
    stem = img_path.stem

    out_img_path = os.path.join(DATASET_OUT_DIR, split_name, "images", img_name)
    if os.path.exists(out_img_path):
        img_name = f"{stem}_{idx}{img_path.suffix}"
        stem = f"{stem}_{idx}"
        out_img_path = os.path.join(DATASET_OUT_DIR, split_name, "images", img_name)
    shutil.copy2(img_path, out_img_path)

    seg_map, segments_info = predict_instances(image)
    gt_instance_mask = np.zeros((image.height, image.width), dtype=np.uint16)
    img_polys = []

    inst_counter = 1
    for seg in segments_info:
        inst_id = seg["id"]
        mask = seg_map == inst_id
        area_px = int(mask.sum())
        if area_px < MIN_POLYGON_AREA_PX:
            continue

        poly_norm = mask_to_normalized_polygon(mask, image.width, image.height, MIN_POLYGON_AREA_PX)
        if poly_norm is None:
            continue

        ade_cls_id = int(seg["label_id"])
        label_name = ade_id2label.get(ade_cls_id, f"class_{ade_cls_id}")
        detected_ade_ids.add(ade_cls_id)

        gt_instance_mask[mask] = inst_counter
        inst_counter += 1

        ys, xs = np.where(mask)
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

        img_polys.append((ade_cls_id, poly_norm))
        raw_records.append({
            "split": split_name,
            "image_name": img_name,
            "width": image.width,
            "height": image.height,
            "instance_id": inst_counter - 1,
            "ade_label_id": ade_cls_id,
            "label_name": label_name,
            "score": round(float(seg["score"]), 4),
            "area_px": area_px,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
        })

    mask_save_arr = gt_instance_mask.astype(np.uint8) if gt_instance_mask.max() <= 255 else gt_instance_mask
    Image.fromarray(mask_save_arr).save(os.path.join(DATASET_OUT_DIR, split_name, "masks", f"{stem}_mask.png"))

    lbl_out_path = os.path.join(DATASET_OUT_DIR, split_name, "labels", f"{stem}.txt")
    pending_label_files.append((lbl_out_path, img_polys))

    vis_img = draw_instance_segmentation(image, seg_map, segments_info)
    if SAVE_ALL_VISUALIZATIONS:
        vis_img.save(os.path.join(VIS_DIR, split_name, f"vis_{img_name}"))
    if len(visualized_samples) < MAX_VIS_IMAGES:
        visualized_samples.append((split_name, img_name, image, vis_img, len(img_polys)))

# Susun pemetaan kelas & buat data.yaml dari output model Swin-Large
if REMAP_TO_DETECTED_CLASSES and len(detected_ade_ids) > 0:
    sorted_ade_ids = sorted(detected_ade_ids)
    ade_to_final_id = {ade_id: new_id for new_id, ade_id in enumerate(sorted_ade_ids)}
    final_names = {new_id: ade_id2label[ade_id] for new_id, ade_id in enumerate(sorted_ade_ids)}
else:
    ade_to_final_id = {ade_id: ade_id for ade_id in ade_id2label}
    final_names = dict(ade_id2label)

for lbl_out_path, img_polys in pending_label_files:
    lines = []
    for ade_cls_id, poly_norm in img_polys:
        final_cls_id = ade_to_final_id[ade_cls_id]
        coord_str = " ".join(f"{v:.6f}" for v in poly_norm)
        lines.append(f"{final_cls_id} {coord_str}")
    with open(lbl_out_path, "w", encoding="utf-8") as lf:
        lf.write("\n".join(lines))

df_results = pd.DataFrame(raw_records)
if not df_results.empty:
    df_results["class_id"] = df_results["ade_label_id"].map(ade_to_final_id)
df_results.to_csv(CSV_OUTPUT_PATH, index=False)

generated_yaml_payload = {
    "path": DATASET_OUT_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(final_names),
    "names": final_names,
}
for yaml_target in [DATA_YAML_WORKING, DATA_YAML_OUTPUT, DATA_YAML_DATASET]:
    with open(yaml_target, "w", encoding="utf-8") as yf:
        yaml.safe_dump(generated_yaml_payload, yf, sort_keys=False)

zip_file_path = shutil.make_archive(ZIP_OUTPUT_BASE, "zip", OUTPUT_DIR)
zip_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
elapsed = time.time() - start_time

print(f"Selesai dalam {elapsed:.2f} detik ({len(image_split_records)} gambar).")
print(f"Total instance terdeteksi : {len(df_results)}")
print(f"Jumlah kelas di data.yaml : {len(final_names)}")
print(f"data.yaml tersimpan di    : {DATA_YAML_WORKING}")
print(f"CSV hasil tersimpan di    : {CSV_OUTPUT_PATH}")
print(f"File ZIP Kaggle Output    : {zip_file_path} ({zip_size_mb:.2f} MB)")


### Preview Input vs Ground Truth Hasil Swin-Large & Isi `data.yaml`


In [ ]:
if len(visualized_samples) > 0:
    for split_name, name, orig_img, vis_img, n_inst in visualized_samples:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(orig_img)
        axes[0].set_title(f"[{split_name}] Input: {name}", fontsize=11)
        axes[0].axis("off")

        axes[1].imshow(vis_img)
        axes[1].set_title(f"Generated GT Swin-Large ({n_inst} instances)", fontsize=11)
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()


In [ ]:
with open(DATA_YAML_WORKING, "r", encoding="utf-8") as yf:
    print("=== Isi data.yaml yang dihasilkan dari Swin-Large ===")
    print(yf.read())

if not df_results.empty:
    display(df_results.head(10))
    print("\nTop 15 Kelas yang Terdeteksi:")
    display(
        df_results["label_name"]
        .value_counts()
        .head(15)
        .rename_axis("label_name")
        .reset_index(name="count")
    )


## 6. (Opsional) Fine-Tuning Menggunakan Dataset & `data.yaml` Hasil Generasi Swin-Large
Bagian ini dapat langsung menggunakan `DATA_YAML_OUTPUT` dan `DATASET_OUT_DIR` yang baru saja dihasilkan pada langkah sebelumnya.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class GeneratedYoloPolyDataset(Dataset):
    def __init__(self, images_dir: str, labels_dir: str, processor, image_size=(512, 512)):
        self.image_paths = sorted(
            [p for p in Path(images_dir).glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
        )
        self.labels_dir = Path(labels_dir)
        self.processor = processor
        self.image_size = image_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB").resize(self.image_size)
        w, h = image.size

        instance_mask = np.full((h, w), 255, dtype=np.int32)
        inst2sem = {}

        label_file = self.labels_dir / f"{img_path.stem}.txt"
        if label_file.exists():
            with open(label_file, "r", encoding="utf-8") as f:
                lines = [ln.strip() for ln in f if ln.strip()]

            inst_idx = 1
            for line in lines:
                parts = line.split()
                if len(parts) < 7:
                    continue
                cls_id = int(float(parts[0]))
                coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                coords[:, 0] = np.clip(coords[:, 0] * w, 0, w - 1)
                coords[:, 1] = np.clip(coords[:, 1] * h, 0, h - 1)
                poly = np.round(coords).astype(np.int32)
                if inst_idx == 255:
                    inst_idx += 1
                cv2.fillPoly(instance_mask, [poly], color=inst_idx)
                inst2sem[inst_idx] = cls_id
                inst_idx += 1

        encoded = self.processor(
            images=image,
            segmentation_maps=instance_mask,
            instance_id_to_semantic_id=inst2sem,
            return_tensors="pt",
        )
        return {k: v.squeeze(0) if isinstance(v, torch.Tensor) else v[0] for k, v in encoded.items()}


def mask2former_collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "pixel_mask": torch.stack([b["pixel_mask"] for b in batch]),
        "mask_labels": [b["mask_labels"] for b in batch],
        "class_labels": [b["class_labels"] for b in batch],
    }


In [ ]:
RUN_FINETUNE = False
EPOCHS = 5
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-5
FINETUNE_SAVE_DIR = os.path.join(WORKING_DIR, "mask2former-finetuned")

if RUN_FINETUNE:
    with open(DATA_YAML_OUTPUT, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)

    names = data_cfg["names"]
    if isinstance(names, dict):
        custom_id2label = {int(k): str(v) for k, v in names.items()}
    else:
        custom_id2label = {i: str(n) for i, n in enumerate(names)}
    custom_label2id = {v: k for k, v in custom_id2label.items()}

    ft_processor = AutoImageProcessor.from_pretrained(MODEL_NAME, do_reduce_labels=False, ignore_index=255)
    ft_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        MODEL_NAME,
        id2label=custom_id2label,
        label2id=custom_label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    train_dataset = GeneratedYoloPolyDataset(
        images_dir=os.path.join(DATASET_OUT_DIR, "train", "images"),
        labels_dir=os.path.join(DATASET_OUT_DIR, "train", "labels"),
        processor=ft_processor,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=mask2former_collate_fn,
        num_workers=2,
    )

    optimizer = torch.optim.AdamW(ft_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    ft_model.train()
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}")
        for step, batch in pbar:
            pixel_values = batch["pixel_values"].to(device)
            pixel_mask = batch["pixel_mask"].to(device)
            mask_labels = [m.to(device) for m in batch["mask_labels"]]
            class_labels = [c.to(device) for c in batch["class_labels"]]

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                outputs = ft_model(
                    pixel_values=pixel_values,
                    pixel_mask=pixel_mask,
                    mask_labels=mask_labels,
                    class_labels=class_labels,
                )
                loss = outputs.loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += outputs.loss.item()
            pbar.set_postfix({"loss": f"{total_loss / (step + 1):.4f}"})

    os.makedirs(FINETUNE_SAVE_DIR, exist_ok=True)
    ft_model.save_pretrained(FINETUNE_SAVE_DIR)
    ft_processor.save_pretrained(FINETUNE_SAVE_DIR)
    print(f"Model hasil fine-tuning disimpan ke: {FINETUNE_SAVE_DIR}")
else:
    print("Fine-tuning dilewati (RUN_FINETUNE = False).")
